In [ ]:
!pip install mlflow

In [ ]:
!pip install dagshub

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_PATH = '/content/drive/MyDrive/ML final project'

if not os.path.exists(PROJECT_PATH):
    os.makedirs(PROJECT_PATH)
    print(f"it created: {PROJECT_PATH}")

%cd {PROJECT_PATH}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/ML final project


In [ ]:
!pip install darts pytorch-lightning matplotlib pandas numpy

In [ ]:
import os
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.base import BaseEstimator, TransformerMixin
from walmart_transformers import WalmartLagTransformer_v2


In [ ]:
YOUR_DAGSHUB_USERNAME = "mkekn23"
YOUR_DAGSHUB_TOKEN = ""

REPO_OWNER = "mesata"
REPO_NAME = "Walmart---Store-Sales-Forecasting"

os.environ['MLFLOW_TRACKING_USERNAME'] = YOUR_DAGSHUB_USERNAME
os.environ['MLFLOW_TRACKING_PASSWORD'] = YOUR_DAGSHUB_TOKEN

mlflow.set_tracking_uri(f"https://dagshub.com/{REPO_OWNER}/{REPO_NAME}.mlflow")


runs = mlflow.search_runs(
    experiment_ids=["0"],
    filter_string="tags.mlflow.runName = 'XGBoost_with_Many_Lags_parameters'"
)

if runs.empty:
    raise ValueError(" მოდელი ამ სახელით ვერ მოიძებნა! შეამოწმე სახელი MLflow-ში.")


model_registry_uri = "models:/XGBoost_best/1"

print(f"📦 მოდელის ჩამოტვირთვა რეესტრიდან: {model_registry_uri} ...")

loaded_pipeline = mlflow.sklearn.load_model(model_registry_uri)
print(" მოდელი და მთლიანი პაიპლაინი წარმატებით ჩამოიტვირთა რეესტრიდან!")


📦 მოდელის ჩამოტვირთვა რეესტრიდან: models:/XGBoost_best/1 ...


 მოდელი და მთლიანი პაიპლაინი წარმატებით ჩამოიტვირთა რეესტრიდან!


In [ ]:
test_df = pd.read_csv('/content/drive/MyDrive/ML final project/walmart_data/test.csv.zip')
train_df = pd.read_csv('/content/drive/MyDrive/ML final project/walmart_data/train.csv.zip')

In [ ]:
# ==========================================
# FINAL SUBMISSION BLOCK — თავისუფალი ცვლადი, ძველი cell-ები ვერ ჩაერევა
# ==========================================
import pandas as pd

def build_submission_features(test_df, train_df):
    df = test_df.copy()
    df['Date'] = pd.to_datetime(df['Date'])

    hist = pd.concat([
        train_df[['Store', 'Dept', 'Date', 'Weekly_Sales']].assign(
            Date=lambda d: pd.to_datetime(d['Date'])
        ),
        df[['Store', 'Dept', 'Date']].assign(Weekly_Sales=pd.NA)
    ], ignore_index=True)

    hist = hist.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)

    hist['Lag_51'] = hist.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(51)
    hist['Lag_52'] = hist.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(52)
    hist['Lag_53'] = hist.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(53)
    hist['Rolling_Mean_4_Yearly'] = (
        hist.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(52).rolling(window=4).mean()
    )
    hist['Max_Sales_Yearly_Window'] = (
        hist.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(51).rolling(window=3, center=True).max()
    )

    lag_cols = ['Lag_51', 'Lag_52', 'Lag_53', 'Rolling_Mean_4_Yearly', 'Max_Sales_Yearly_Window']
    lookup = hist[['Store', 'Dept', 'Date'] + lag_cols].drop_duplicates(subset=['Store', 'Dept', 'Date'])

    out = df.merge(lookup, on=['Store', 'Dept', 'Date'], how='left')
    out[lag_cols] = out[lag_cols].fillna(0)
    return out

final_test_features = build_submission_features(test_df, train_df)

print("სვეტები:", final_test_features.columns.tolist())
assert 'Date' in final_test_features.columns
assert 'Date_x' not in final_test_features.columns
print("✅ სისუფთავე დადასტურდა")

final_predictions = loaded_pipeline.predict(final_test_features)
final_predictions = final_predictions.clip(min=0)

submission = pd.DataFrame({
    'Id': (
        final_test_features['Store'].astype(str) + '_' +
        final_test_features['Dept'].astype(str) + '_' +
        final_test_features['Date'].dt.strftime('%Y-%m-%d')
    ),
    'Weekly_Sales': final_predictions
})

submission.to_csv('submission.csv', index=False)
print("✅ submission.csv წარმატებით შეიქმნა!")
print(submission.head())
print(f"სულ სტრიქონი: {len(submission)}")

/tmp/ipykernel_1035/42477463.py:10: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  hist = pd.concat([


სვეტები: ['Store', 'Dept', 'Date', 'IsHoliday', 'Lag_51', 'Lag_52', 'Lag_53', 'Rolling_Mean_4_Yearly', 'Max_Sales_Yearly_Window']
✅ სისუფთავე დადასტურდა
✅ submission.csv წარმატებით შეიქმნა!
               Id  Weekly_Sales
0  1_1_2012-11-02  39756.808594
1  1_1_2012-11-09  22521.140625
2  1_1_2012-11-16  22556.064453
3  1_1_2012-11-23  24889.699219
4  1_1_2012-11-30  27245.750000
სულ სტრიქონი: 115064


-----------------------------------------------------------------------------------------------------

In [ ]:
# გავუშვათ build_submission_features და ვნახოთ NaN-ების პროცენტი fillna-მდე
def build_submission_features_debug(test_df, train_df):
    df = test_df.copy()
    df['Date'] = pd.to_datetime(df['Date'])

    hist = pd.concat([
        train_df[['Store', 'Dept', 'Date', 'Weekly_Sales']].assign(
            Date=lambda d: pd.to_datetime(d['Date'])
        ),
        df[['Store', 'Dept', 'Date']].assign(Weekly_Sales=pd.NA)
    ], ignore_index=True)

    hist = hist.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)

    hist['Lag_51'] = hist.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(51)
    hist['Lag_52'] = hist.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(52)
    hist['Lag_53'] = hist.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(53)
    hist['Rolling_Mean_4_Yearly'] = (
        hist.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(52).rolling(window=4).mean()
    )
    hist['Max_Sales_Yearly_Window'] = (
        hist.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(51).rolling(window=3, center=True).max()
    )

    lag_cols = ['Lag_51', 'Lag_52', 'Lag_53', 'Rolling_Mean_4_Yearly', 'Max_Sales_Yearly_Window']
    lookup = hist[['Store', 'Dept', 'Date'] + lag_cols].drop_duplicates(subset=['Store', 'Dept', 'Date'])

    out = df.merge(lookup, on=['Store', 'Dept', 'Date'], how='left')
    return out, lag_cols


test_df_clean = pd.read_csv('/content/drive/MyDrive/ML final project/walmart_data/test.csv.zip')
debug_features, lag_cols = build_submission_features_debug(test_df_clean, train_df)

print("NaN პროცენტი fillna-მდე:")
print(debug_features[lag_cols].isna().mean() * 100)

/tmp/ipykernel_1035/4289147033.py:6: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  hist = pd.concat([


NaN პროცენტი fillna-მდე:
Lag_51                     1.134151
Lag_52                     1.169784
Lag_53                     1.205416
Rolling_Mean_4_Yearly      1.278419
Max_Sales_Yearly_Window    3.791803
dtype: float64


In [ ]:
def build_submission_features_v2(test_df, train_df):
    df = test_df.copy()
    df['Date'] = pd.to_datetime(df['Date'])

    hist = pd.concat([
        train_df[['Store', 'Dept', 'Date', 'Weekly_Sales']].assign(
            Date=lambda d: pd.to_datetime(d['Date'])
        ),
        df[['Store', 'Dept', 'Date']].assign(Weekly_Sales=pd.NA)
    ], ignore_index=True)

    hist = hist.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)

    hist['Lag_51'] = hist.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(51)
    hist['Lag_52'] = hist.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(52)
    hist['Lag_53'] = hist.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(53)
    hist['Rolling_Mean_4_Yearly'] = (
        hist.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(52).rolling(window=4).mean()
    )
    hist['Max_Sales_Yearly_Window'] = (
        hist.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(51).rolling(window=3, center=True).max()
    )

    lag_cols = ['Lag_51', 'Lag_52', 'Lag_53', 'Rolling_Mean_4_Yearly', 'Max_Sales_Yearly_Window']
    lookup = hist[['Store', 'Dept', 'Date'] + lag_cols].drop_duplicates(subset=['Store', 'Dept', 'Date'])

    out = df.merge(lookup, on=['Store', 'Dept', 'Date'], how='left')

    # ჭკვიანი fillna: NaN-ები ჯერ Lag_52-ით ვავსოთ (ყველაზე საიმედო proxy),
    # რაც კიდევ NaN დარჩება — Store/Dept მედიანით, და საბოლოოდ 0-ით
    for col in lag_cols:
        out[col] = out[col].fillna(out['Lag_52'])

    store_dept_median = train_df.groupby(['Store', 'Dept'])['Weekly_Sales'].median()
    for col in lag_cols:
        mask = out[col].isna()
        if mask.any():
            fallback = out.loc[mask].set_index(['Store', 'Dept']).index.map(store_dept_median)
            out.loc[mask, col] = fallback.values

    out[lag_cols] = out[lag_cols].fillna(0)  # საბოლოო უსაფრთხოება
    return out

final_test_features_v2 = build_submission_features_v2(test_df_clean, train_df)

print("NaN-ები fix-ის შემდეგ:\n", final_test_features_v2[['Lag_51','Lag_52','Lag_53','Rolling_Mean_4_Yearly','Max_Sales_Yearly_Window']].isna().sum())

/tmp/ipykernel_1035/2108712009.py:5: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  hist = pd.concat([


NaN-ები fix-ის შემდეგ:
 Lag_51                     0
Lag_52                     0
Lag_53                     0
Rolling_Mean_4_Yearly      0
Max_Sales_Yearly_Window    0
dtype: int64


In [ ]:
from walmart_transformers import WalmartDataTransformer_updatedFeatureEngineering, TimeSeriesSplitter, WalmartLagTransformer

train_df_lags = train_df.copy()
train_df_lags['Lag_51'] = train_df.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(51)
train_df_lags['Lag_52'] = train_df.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(52)
train_df_lags['Lag_53'] = train_df.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(53)
train_df_lags['Rolling_Mean_4_Yearly'] = train_df.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(52).rolling(window=4).mean()

train_df_lags = train_df_lags.dropna(subset=['Lag_52', 'Rolling_Mean_4_Yearly'])

train_df_lags['Max_Sales_Yearly_Window'] = (
    train_df.groupby(['Store', 'Dept'])['Weekly_Sales']
    .shift(51)
    .rolling(window=3, center=True)
    .max()
)

splitter = TimeSeriesSplitter(split_date='2012-01-01', target_col='Weekly_Sales')
X_train_raw, y_train, X_val_raw, y_val = splitter.split(train_df_lags)

val_preds = loaded_pipeline.predict(X_val_raw)

val_check = X_val_raw[['IsHoliday']].copy()
val_check['y_true'] = y_val.values
val_check['y_pred'] = val_preds
val_check['abs_error'] = (val_check['y_true'] - val_check['y_pred']).abs()

print(val_check.groupby('IsHoliday')['abs_error'].agg(['mean', 'median', 'count']))

                  mean      median   count
IsHoliday                                 
False      1900.171013  807.457097  118774
True       2040.477930  860.712891    5801


In [ ]:
val_check['signed_error'] = val_check['y_true'] - val_check['y_pred']

print("საშუალო signed error (დადებითი = underpredict, უარყოფითი = overpredict):")
print(val_check.groupby('IsHoliday')['signed_error'].agg(['mean', 'median']))

print("\nWMAE გამოთვლა holiday წონით (5x):")
weights = val_check['IsHoliday'].apply(lambda x: 5 if x else 1)
wmae_before = (val_check['abs_error'] * weights).sum() / weights.sum()
print(f"WMAE (correction-ის გარეშე): {wmae_before:.2f}")

საშუალო signed error (დადებითი = underpredict, უარყოფითი = overpredict):
                 mean      median
IsHoliday                        
False      370.513502  166.591592
True       858.634191  296.662983

WMAE გამოთვლა holiday წონით (5x):
WMAE (correction-ის გარეშე): 1927.71


In [ ]:
import numpy as np

def compute_wmae(y_true, y_pred, is_holiday, holiday_weight=5):
    weights = np.where(is_holiday, holiday_weight, 1)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

# საბაზისო (correction-ის გარეშე)
baseline_wmae = compute_wmae(val_check['y_true'], val_check['y_pred'], val_check['IsHoliday'])
print(f"საბაზისო WMAE: {baseline_wmae:.2f}")

# --- ვცდით additive კორექციას (holiday კვირებზე ვამატებთ მუდმივ რიცხვს) ---
best_additive_wmae = baseline_wmae
best_additive_val = 0
for offset in range(0, 1200, 50):
    adj_preds = val_check['y_pred'] + val_check['IsHoliday'].astype(int) * offset
    wmae = compute_wmae(val_check['y_true'], adj_preds, val_check['IsHoliday'])
    if wmae < best_additive_wmae:
        best_additive_wmae = wmae
        best_additive_val = offset

print(f"\nსაუკეთესო additive offset: +{best_additive_val} → WMAE: {best_additive_wmae:.2f}")

# --- ვცდით multiplicative კორექციას (holiday კვირებზე ვამრავლებთ კოეფიციენტზე) ---
best_mult_wmae = baseline_wmae
best_mult_val = 1.0
for factor in np.arange(1.0, 1.5, 0.02):
    adj_preds = val_check['y_pred'] * np.where(val_check['IsHoliday'], factor, 1.0)
    wmae = compute_wmae(val_check['y_true'], adj_preds, val_check['IsHoliday'])
    if wmae < best_mult_wmae:
        best_mult_wmae = wmae
        best_mult_val = factor

print(f"საუკეთესო multiplicative factor: x{best_mult_val:.2f} → WMAE: {best_mult_wmae:.2f}")

print(f"\n=== შედარება ===")
print(f"Baseline:      {baseline_wmae:.2f}")
print(f"Additive:      {best_additive_wmae:.2f} (გაუმჯობესება: {baseline_wmae - best_additive_wmae:.2f})")
print(f"Multiplicative:{best_mult_wmae:.2f} (გაუმჯობესება: {baseline_wmae - best_mult_wmae:.2f})")

საბაზისო WMAE: 1927.71

საუკეთესო additive offset: +300 → WMAE: 1920.85
საუკეთესო multiplicative factor: x1.04 → WMAE: 1901.83

=== შედარება ===
Baseline:      1927.71
Additive:      1920.85 (გაუმჯობესება: 6.86)
Multiplicative:1901.83 (გაუმჯობესება: 25.88)


In [ ]:
# --- დახვეწილი ძებნა multiplicative factor-ისთვის (თხელი ბადე) ---
best_mult_wmae = baseline_wmae
best_mult_val = 1.0
for factor in np.arange(1.0, 1.15, 0.005):
    adj_preds = val_check['y_pred'] * np.where(val_check['IsHoliday'], factor, 1.0)
    wmae = compute_wmae(val_check['y_true'], adj_preds, val_check['IsHoliday'])
    if wmae < best_mult_wmae:
        best_mult_wmae = wmae
        best_mult_val = factor

print(f"დახვეწილი საუკეთესო factor: x{best_mult_val:.3f} → WMAE: {best_mult_wmae:.2f}")

# --- კომბინირებული: multiplicative + additive ერთად ---
best_combo_wmae = baseline_wmae
best_combo = (1.0, 0)
for factor in np.arange(1.0, 1.12, 0.01):
    for offset in range(0, 400, 25):
        adj_preds = val_check['y_pred'] * np.where(val_check['IsHoliday'], factor, 1.0) + val_check['IsHoliday'].astype(int) * offset
        wmae = compute_wmae(val_check['y_true'], adj_preds, val_check['IsHoliday'])
        if wmae < best_combo_wmae:
            best_combo_wmae = wmae
            best_combo = (factor, offset)

print(f"საუკეთესო კომბო: x{best_combo[0]:.2f} + {best_combo[1]} → WMAE: {best_combo_wmae:.2f}")

print(f"\n=== საბოლოო შედარება ===")
print(f"Baseline:              {baseline_wmae:.2f}")
print(f"Multiplicative (fine): {best_mult_wmae:.2f}")
print(f"Combo:                 {best_combo_wmae:.2f}")

დახვეწილი საუკეთესო factor: x1.050 → WMAE: 1900.95
საუკეთესო კომბო: x1.05 + 0 → WMAE: 1900.95

=== საბოლოო შედარება ===
Baseline:              1927.71
Multiplicative (fine): 1900.95
Combo:                 1900.95


-------------------------------


In [ ]:
# საბოლოო predictions x1.05 კორექციით holiday-კვირებზე
HOLIDAY_FACTOR = 1.05

final_predictions_v2 = loaded_pipeline.predict(final_test_features_v2)
final_predictions_v2 = final_predictions_v2 * np.where(final_test_features_v2['IsHoliday'], HOLIDAY_FACTOR, 1.0)
final_predictions_v2 = final_predictions_v2.clip(min=0)

submission_v2 = pd.DataFrame({
    'Id': (
        final_test_features_v2['Store'].astype(str) + '_' +
        final_test_features_v2['Dept'].astype(str) + '_' +
        final_test_features_v2['Date'].dt.strftime('%Y-%m-%d')
    ),
    'Weekly_Sales': final_predictions_v2
})

submission_v2.to_csv('submission_v2.csv', index=False)
print("✅ submission_v2.csv შეიქმნა!")
print(submission_v2.head())
print(f"სულ სტრიქონი: {len(submission_v2)}")

# შედარებისთვის — რამდენი სტრიქონია holiday
print(f"\nHoliday სტრიქონები test-ში: {final_test_features_v2['IsHoliday'].sum()}")

✅ submission_v2.csv შეიქმნა!
               Id  Weekly_Sales
0  1_1_2012-11-02  39756.808594
1  1_1_2012-11-09  22521.140625
2  1_1_2012-11-16  22556.064453
3  1_1_2012-11-23  26134.184180
4  1_1_2012-11-30  27245.750000
სულ სტრიქონი: 115064

Holiday სტრიქონები test-ში: 8928


In [ ]:
import os
print(os.path.exists('submission_v2.csv'))
print(os.path.getmtime('submission_v2.csv') if os.path.exists('submission_v2.csv') else "ფაილი არ არსებობს")

# შევადაროთ ორივე ფაილი ერთმანეთს
import pandas as pd
old_sub = pd.read_csv('submission.csv')
new_sub = pd.read_csv('submission_v2.csv') if os.path.exists('submission_v2.csv') else None

if new_sub is not None:
    diff = (old_sub['Weekly_Sales'] - new_sub['Weekly_Sales']).abs()
    print(f"საშუალო სხვაობა: {diff.mean():.2f}")
    print(f"მაქსიმალური სხვაობა: {diff.max():.2f}")
    print(f"სტრიქონები სადაც სხვაობა 0-ია: {(diff == 0).sum()} / {len(diff)}")

True
1783748439.0
საშუალო სხვაობა: 140.31
მაქსიმალური სხვაობა: 25845.24
სტრიქონები სადაც სხვაობა 0-ია: 2901 / 115064
